# 04 · Transformations & Joins

> **Module 2 · Spark Fundamentals · Lecture 2.4**

In this notebook you will:

- Project and enrich columns with `select`, `selectExpr`, `withColumn`, and `F.when`/`otherwise`
- Clean strings with the built-in function library instead of UDFs
- Aggregate with `groupBy`/`agg` and combine datasets with `unionByName`
- Use every join type, `F.broadcast`, and window functions (`row_number`, running total, `lag`/`lead`)
- Deduplicate deterministically, and control output with `orderBy`, `sortWithinPartitions`, `repartition`, and `coalesce`

---


## Setup / prerequisites

- **Python 3.9–3.11** and **Java 11 or 17** on your machine.
- **PySpark 3.5.x**: `pip install "pyspark==3.5.1" pandas pyarrow numpy`
- This notebook is **self-contained and offline**: every code cell runs top-to-bottom with
  no GCP account. All data operations use local paths under `/tmp/`.
- GCS / BigQuery / Pub/Sub / JDBC code appears in **clearly-marked optional cells** that
  you uncomment after authenticating with `gcloud auth application-default login`.
- On Dataproc, **do not `pip install pyspark`** — use the Spark runtime already on the
  image (Spark 3.5 on Dataproc 2.2).


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pyspark

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("notebook-01")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("PySpark :", pyspark.__version__)
print("Spark   :", spark.version)
print("Master  :", spark.sparkContext.master)


In [ ]:
# ------------------------------------------------------------------
# Self-contained sample data: orders + customer/product dimensions.
# ------------------------------------------------------------------
orders_rows = [
    ("o-1", "c-1", "p-1", 2, 100.50, "2026-10-01", "EMEA"),
    ("o-2", "c-2", "p-2", 1,  80.00, "2026-10-01", "APAC"),
    ("o-3", "c-1", "p-2", 3,  49.50, "2026-10-02", "EMEA"),
    ("o-4", "c-3", "p-1", 1, 129.00, "2026-10-02", "AMER"),
    ("o-5", "c-2", "p-1", 5,  19.99, "2026-10-03", "APAC"),
    ("o-6", "c-4", "p-3", 2,  49.50, "2026-10-03", "AMER"),
    ("o-7", "c-1", "p-1", 1, 100.50, "2026-10-03", "EMEA"),
]
customers_rows = [
    ("c-1", "Alice", "GOLD"),
    ("c-2", "Bob",   "SILVER"),
    ("c-3", "Carol", "BRONZE"),
]
products_rows = [
    ("p-1", "Widget", "Gadgets"),
    ("p-2", "Gadget", "Gadgets"),
    ("p-3", "Gizmo",  "Misc"),
]

orders = spark.createDataFrame(
    orders_rows,
    schema="order_id string, customer_id string, product_id string, quantity int, "
           "amount double, order_date string, region string",
)
customers = spark.createDataFrame(
    customers_rows, schema="customer_id string, customer_name string, loyalty_tier string",
)
products = spark.createDataFrame(
    products_rows, schema="product_id string, product_name string, category string",
)
orders.show(truncate=False)

## 1 · `select`, `selectExpr`, `withColumn`

| Method | Projection style | Adds or replaces? |
|---|---|---|
| `select(...)` | Column objects | Replaces the whole projection |
| `selectExpr(...)` | SQL strings | Replaces the whole projection |
| `withColumn(name, expr)` | Column objects | Adds or replaces **one** column |
| `withColumns(**kwargs)` | Column objects | Adds or replaces many (Spark 3.2+) |

In [ ]:
projected = orders.select("order_id", "customer_id", "amount")
print("== select ==")
projected.show(truncate=False)

sql_projected = orders.selectExpr(
    "order_id",
    "customer_id",
    "amount",
    "amount * quantity AS line_value",
    "upper(region) AS region_upper",
)
print("\n== selectExpr ==")
sql_projected.show(truncate=False)

enriched = orders.withColumn("line_value", F.col("amount") * F.col("quantity"))
print("\n== withColumn (keeps everything else) ==")
enriched.show(truncate=False)

## 2 · Conditional logic with `F.when` / `otherwise`

Semantics are **first-match-wins, in order**, exactly like a SQL `CASE WHEN` chain.
A `when` chain without `otherwise` yields `null` for non-matching rows — silently.

In [ ]:
tiered = (
    orders
    .withColumn(
        "amount_band",
        F.when(F.col("amount") >= F.lit(100), F.lit("high"))
         .when(F.col("amount") >= F.lit(50),  F.lit("medium"))
         .otherwise(F.lit("low")),
    )
    .withColumn(
        "is_priority",
        F.when(F.col("amount") > F.lit(100), F.lit(True))
         .otherwise(F.lit(False)),
    )
)
tiered.select("order_id", "amount", "amount_band", "is_priority").show(truncate=False)

## 3 · String functions

Use the built-in library. A Python UDF is an opaque box to Catalyst — it blocks
predicate pushdown and costs one interpreter hop per row.

| Task | Function |
|---|---|
| Trim / case | `F.trim`, `F.lower`, `F.upper` |
| Split | `F.split(col, r"\s+")`, `.getItem(0)` |
| Pattern match | `F.regexp_extract`, `F.regexp_replace`, `F.contains` |
| Concatenate | `F.concat` (null if any arg is null) vs `F.concat_ws` (skips nulls) |
| Pad | `F.lpad`, `F.rpad` |

In [ ]:
strings = (
    orders
    .withColumn("region_clean", F.lower(F.trim(F.col("region"))))
    .withColumn("region_padded", F.lpad(F.col("region_clean"), 6, "-"))
    .withColumn("has_emea", F.contains(F.lower(F.col("region")), F.lit("emea")))
    .withColumn("digits", F.regexp_replace(F.col("order_id"), r"\D", ""))
    .withColumn("region_ws", F.concat_ws("-", F.col("region"), F.col("customer_id")))
)
strings.select("order_id", "region_clean", "region_padded", "has_emea", "digits", "region_ws").show(truncate=False)

## 4 · `groupBy` / `agg`

`F.sum`/`F.avg` ignore nulls, `F.count("*")` counts every row, `F.count("col")` counts
only non-null values, and `F.countDistinct` is exact but expensive (it forces a second
aggregation pass).

**Caution:** `df.agg(...)` computes a global aggregate **without** the extra shuffle that
`df.groupBy().agg(...)` incurs. Use `agg` for whole-table totals.

In [ ]:
by_region = (
    orders
    .groupBy("region")
    .agg(
        F.count("*").alias("orders"),
        F.sum("amount").alias("revenue"),
        F.avg("amount").alias("avg_amount"),
        F.countDistinct("customer_id").alias("customers"),
        F.max("order_date").alias("last_order"),
    )
    .orderBy(F.desc("revenue"))
)
by_region.show(truncate=False)

print("global aggregate (no shuffle):")
orders.agg(
    F.count("*").alias("orders"),
    F.sum("amount").alias("revenue"),
).show()

## 5 · Joins — all types

```
   LEFT rows                RIGHT rows

   [1]----\
   [2]-----\                /----[2]
   [3]----- >--- JOIN ------ >----[4]
   [4]----/                \----[5]
   [5]----/                  \---[6]

   inner   : rows matching on both sides
   left    : all left rows, nulls on the right
   right   : all right rows, nulls on the left
   outer   : all rows from both sides
   semi    : left rows WITH a match (left columns only)
   anti    : left rows WITHOUT a match (left columns only)
   cross   : cartesian product
```

**Note:** `left_semi` and `left_anti` are the efficient existence check and anti-join — they
never carry the right side's payload, so they are faster than `left` + `filter`.

In [ ]:
dim = customers.withColumnRenamed("customer_id", "customer_id")

join_results = {
    "inner":   orders.join(dim, "customer_id", "inner"),
    "left":    orders.join(dim, "customer_id", "left"),
    "right":   orders.join(dim, "customer_id", "right"),
    "outer":   orders.join(dim, "customer_id", "outer"),
    "semi":    orders.join(dim, "customer_id", "left_semi"),
    "anti":    orders.join(dim, "customer_id", "left_anti"),
}
for name, df in join_results.items():
    print(f"{name:7s} rows = {df.count()}")

print("\n== left join (c-4 has no dimension row) ==")
orders.join(dim, "customer_id", "left").select(
    "order_id", "customer_id", "customer_name", "loyalty_tier"
).show(truncate=False)

print("== left_anti (orphans) ==")
orders.join(dim, "customer_id", "left_anti").select("order_id", "customer_id").show(truncate=False)

### `F.broadcast`

Broadcasting copies the small side to every executor and eliminates the shuffle for the
join. Rule of thumb: the small side should be **under ~10 MB**.

**Caution:** If the optimizer auto-broadcasts a table that is too large, every executor tries to
build the hash table in memory and the job dies with `OutOfMemoryError` — often after
minutes of work. Pin the strategy with `F.broadcast` when you know the answer.

**Caution:** `spark.executor.memory` and `spark.driver.memory` cannot be changed at runtime;
the broadcast threshold is just a config.

In [ ]:
joined = orders.join(F.broadcast(customers), "customer_id", "inner")
joined.select("order_id", "customer_name", "loyalty_tier", "amount").show(truncate=False)
joined.explain(mode="formatted")

print("autoBroadcastJoinThreshold:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

## 6 · Window functions

A window function computes across related rows **without collapsing them** — the row
count is preserved.

```
Window
  .partitionBy("customer_id")     <- independent groups
  .orderBy("order_date")         <- order within each group
  .rowsBetween(...)              <- which rows the function sees
```

| Frame | Meaning |
|---|---|
| `rowsBetween(unboundedPreceding, currentRow)` | Running total |
| `rowsBetween(-2, 0)` | Current row plus the two before it |
| `rowsBetween(unboundedPreceding, unboundedFollowing)` | The whole partition |

**Caution:** `row_number` is non-deterministic when there are ties — always add a deterministic
tiebreaker to the `orderBy`.

In [ ]:
from pyspark.sql import Window

# Deterministic window: order_date then order_id as a tiebreaker.
cust_window = Window.partitionBy("customer_id").orderBy("order_date", "order_id")

windowed = (
    orders
    .withColumn("rn", F.row_number().over(cust_window))
    .withColumn("prev_amount", F.lag("amount").over(cust_window))
    .withColumn("next_amount", F.lead("amount").over(cust_window))
    .withColumn(
        "running_total",
        F.sum("amount").over(
            cust_window.rowsBetween(Window.unboundedPreceding, Window.currentRow)
        ),
    )
)
windowed.select(
    "order_id", "customer_id", "order_date", "amount",
    "rn", "prev_amount", "next_amount", "running_total",
).orderBy("customer_id", "order_date").show(truncate=False)

## 7 · `dropDuplicates(subset=)`

`dropDuplicates(subset=...)` keeps an **arbitrary** row — there is no ordering
guarantee. When the retained row matters (you want the latest), use `row_number`.

In [ ]:
# Arbitrary row retained.
dedup = orders.dropDuplicates(subset=["customer_id"])
print("dropDuplicates rows:", dedup.count())
dedup.select("order_id", "customer_id").orderBy("customer_id").show(truncate=False)

# Deterministic: keep the latest order per customer.
latest_window = Window.partitionBy("customer_id").orderBy(
    F.col("order_date").desc(), F.col("order_id").desc()
)
latest = (
    orders
    .withColumn("rn", F.row_number().over(latest_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)
print("\nlatest per customer:")
latest.select("order_id", "customer_id", "order_date").orderBy("customer_id").show(truncate=False)

## 8 · `unionByName`

`union` is **positional** — it zips columns by index and ignores names. `unionByName`
matches by name and is tolerant of missing columns (`allowMissingColumns=True`, Spark 3.4+).

**Caution:** Positional `union` with the same column count but different order produces silently
wrong data. Use `unionByName` for anything that arrives over time.

In [ ]:
batch_a = orders.filter(F.col("order_date") == "2026-10-01")
batch_b = orders.filter(F.col("order_date") == "2026-10-02")

print("batch_a:", batch_a.count(), "batch_b:", batch_b.count())
print("unionByName:", batch_a.unionByName(batch_b).count())

# A batch with a new column and a missing one.
batch_c = batch_b.withColumn("channel", F.lit("web")).drop("region")
combined = batch_a.unionByName(batch_c, allowMissingColumns=True)
combined.printSchema()
combined.show(truncate=False)

## 9 · Ordering and partitioning

| Operation | Shuffle? | Notes |
|---|---|---|
| `orderBy(col)` | **Yes** — global sort | One total order across all partitions |
| `sortWithinPartitions(col)` | No | Sort inside each partition only — cheap, but no global order |
| `repartition(n)` / `repartition(n, col)` | **Yes** | Even sizing, or hash-partition by key |
| `coalesce(n)` | No | Fewer output files; distribution may be uneven |

Use `sortWithinPartitions` **after** a `repartition(col)` to get a per-key ordering
without paying for a second global sort.

In [ ]:
print("== orderBy (global) ==")
orders.orderBy(F.desc("amount")).select("order_id", "amount").show(truncate=False)

print("== sortWithinPartitions (per partition) ==")
orders.repartition(2, F.col("region")).sortWithinPartitions(F.desc("amount")) \
      .select("order_id", "region", "amount").show(truncate=False)

print("== repartition vs coalesce ==")
print("base          :", orders.rdd.getNumPartitions())
print("repartition(2):", orders.repartition(2).rdd.getNumPartitions())
print("coalesce(2)   :", orders.coalesce(2).rdd.getNumPartitions())

## 10 · Read the physical plan

`explain(True)` is identical to `explain(mode="formatted")`. Read it bottom-up:

- `Exchange` — a full shuffle. Count them.
- `PushedFilters` under `FileScan` — is the filter reaching the scan?
- `BroadcastHashJoin` — a broadcast join was chosen.
- `SortMergeJoin` — both sides were range-partitioned and sorted.

In [ ]:
enriched = (
    orders
    .join(F.broadcast(customers), "customer_id", "inner")
    .join(F.broadcast(products), "product_id", "inner")
    .withColumn("line_value", F.col("amount") * F.col("quantity"))
)
enriched.groupBy("region", "category").agg(
    F.sum("line_value").alias("revenue"),
    F.count("*").alias("orders"),
).explain(True)

## 🔑 Key Takeaways

- `select` replaces the projection; `withColumn` adds to it.
- `F.when` is first-match-wins; without `otherwise` non-matching rows get `null` silently.
- Prefer the built-in string functions over UDFs — Catalyst cannot optimize past a Python UDF.
- `df.agg(...)` computes a global aggregate without the extra shuffle of `df.groupBy().agg(...)`.
- All six join types behave differently: `left_semi` and `left_anti` are the efficient existence check and anti-join.
- **Broadcast only when the small side is genuinely small (< ~10 MB)** — an oversized auto-broadcast OOMs every executor.
- Window functions preserve row count but require a shuffle for `partitionBy` and a sort for `orderBy`.
- `dropDuplicates(subset=...)` keeps an arbitrary row; use `row_number` with a deterministic tiebreaker when the retained row matters.
- `union` is positional and can silently mis-map columns — use `unionByName`.
- `orderBy` is a global sort; `sortWithinPartitions` is per-partition and cheap.

---

## 📚 Next steps

- Compare the `explain(True)` output of a broadcast join with a plain join on the same data.
- Continue to `05-reading-from-gcs-and-bigquery.ipynb` for multi-source extraction.
